# Zugriff auf HID Geräte aus Python unter Windows und Linux

z.B. die VC 870 Messgeräte an der Hochschule und meine Uni-T Messgeräte haben nur ein hid Interface. Leider keine echte USB Serial Schnittstelle.

Ich suchte eine Lösung die unter Linux und unter Windows gleichermaßen funktioniert.

## Python Paket und die Dokumentation

Das Paket heißt hidapi und es stellt hid und hidraw bereit. Beide mit gleicher Schnittstelle.

- Code https://github.com/trezor/cython-hidapi
- PyPi https://pypi.org/project/hidapi/

Benötigt wird die libusb.

## Windows

https://github.com/libusb/hidapi/releases

Herunterladen und im Python verzeichnis ablegen: C:\Users\mathias.moog\AppData\Local\Programs\Python\Python313

Anschließend das passende Paket installieren:

    pip install hidapi
  
Unter Windows steht nur das hid Interface zur Verfügung, nicht hidraw.

## Linux 

Regeln für die hidraw Geräte anpassen, siehe https://gitlab.com/CalcProgrammer1/OpenRGB/-/work_items/1985

    sudo nano /etc/udev/rules.d/51-usb-device.rules

Folgende Zeile hinzu

    SUBSYSTEMS=="hidraw", GROUP="plugdev", MODE="666"

Damm die Regeln neu laden

    sudo udevadm control --reload-rules

Jetzt wird noch die aktuelle hid implementierung benötigt. 

    sudo apt install python3-hid

Im Python Code

    import hidraw as hid



## hid 

Benötigt hidapi als externes Paket. Siehe pypi: https://pypi.org/project/hidapi/, Dokumentation https://trezor.github.io/cython-hidapi/api.html, Source Code https://github.com/trezor/cython-hidapi

    pip install hidapi

Unter Linux ist es komplett mit Abhängigkeiten als Paket enthalten

    sudo apt install python3-hid

    

In [1]:
import sys
if sys.platform.startswith('linux'):
    import hidraw as hid
else:
    import hid
import pandas as pd

## Geräte auflisten

In [2]:
H = hid.enumerate()
H

[{'path': b'/dev/hidraw2',
  'vendor_id': 6790,
  'product_id': 57352,
  'serial_number': '',
  'release_number': 4864,
  'manufacturer_string': 'WCH.CN \x13',
  'product_string': 'USB to Serial',
  'usage_page': 65440,
  'usage': 1,
  'interface_number': 0,
  'bus_type': 1},
 {'path': b'/dev/hidraw0',
  'vendor_id': 1739,
  'product_id': 52545,
  'serial_number': '',
  'release_number': 0,
  'manufacturer_string': '',
  'product_string': 'SYNA7DB5:01 06CB:CD41',
  'usage_page': 1,
  'usage': 2,
  'interface_number': -1,
  'bus_type': 3},
 {'path': b'/dev/hidraw0',
  'vendor_id': 1739,
  'product_id': 52545,
  'serial_number': '',
  'release_number': 0,
  'manufacturer_string': '',
  'product_string': 'SYNA7DB5:01 06CB:CD41',
  'usage_page': 13,
  'usage': 5,
  'interface_number': -1,
  'bus_type': 3},
 {'path': b'/dev/hidraw0',
  'vendor_id': 1739,
  'product_id': 52545,
  'serial_number': '',
  'release_number': 0,
  'manufacturer_string': '',
  'product_string': 'SYNA7DB5:01 06CB:CD

{'path': b'\\\\?\\HID#VID_1A86&PID_E008#9&2afa1e71&0&0000#{4d1e55b2-f16f-11cf-88cb-001111000030}',
  'vendor_id': 6790,
  'product_id': 57352,
  'serial_number': '',
  'release_number': 4864,
  'manufacturer_string': 'WCH.CN \x13',
  'product_string': 'USB to Serial',
  'usage_page': 65440,
  'usage': 1,
  'interface_number': 0,
  'bus_type': <BusType.USB: 1>}]

In [3]:
Hdf = pd.json_normalize(H)
Hdf["path"] = Hdf["path"].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)
Hdf.drop(columns=['release_number', 'interface_number', 'usage_page', 'usage', 'interface_number', 'bus_type'], inplace=True)
Hdf

,path,vendor_id,product_id,serial_number,manufacturer_string,product_string
0,/dev/hidraw2,6790,57352,,WCH.CN ,USB to Serial
1,/dev/hidraw0,1739,52545,,,SYNA7DB5:01 06CB:CD41
2,/dev/hidraw0,1739,52545,,,SYNA7DB5:01 06CB:CD41
3,/dev/hidraw0,1739,52545,,,SYNA7DB5:01 06CB:CD41
4,/dev/hidraw0,1739,52545,,,SYNA7DB5:01 06CB:CD41
5,/dev/hidraw1,4661,43554,0f:0f:0f:0f:0f:0f,,ID0191


In [4]:
# Den Pfad für eines der Geräte ausgeben, z.B. für den VC870:
Hdf.loc[3]["path"]

'/dev/hidraw0'

### Merker für verschiedene Geräte

- Optische Schnittstelle für VC870 Multimeter melden sich als vendor_id = 6790	product_id 57352		WCH.CN 	USB to Serial
